# Training PerLeadCNN

This notebook trains **PerLeadCNN** with the exact released configuration by
calling the same code as `src/train.py` (so the notebook and the script can
never drift apart). It runs a configurable number of patient-grouped splits
and writes `summary.json`, `per_split.json`, `best_model.pt`, and
`median_model.pt` — the same artifacts that ship in
`results/multisplit_dbb6f49/`.

Requires the ECG dataset (PHI, not shipped — see `DATA.md`).

**Reproducibility note:** training is seeded per split, but exact bit-for-bit
reproduction of the released checkpoints is not guaranteed across different
hardware / BLAS / cuDNN versions. Retraining reproduces the *distribution*
(mean AUROC ~= 0.71 over 30 splits); the shipped checkpoints reproduce the
*exact* recorded metrics — verify those with `reproduce.ipynb`.

In [ ]:
import os, sys
import numpy as np
# Make the package importable when running from the package root.
sys.path.insert(0, os.path.abspath('.'))
# Point at the ECG dataset (PHI, not shipped — see DATA.md). The dataset lives
# at the repo root, one level above this package. Edit this path if yours is
# elsewhere.
os.environ['REPNET_DATA_DIR'] = os.path.abspath(
    os.path.join('..', 'data', 'seniordesign_upload'))
assert os.path.isdir(os.environ['REPNET_DATA_DIR']), \
    f"data dir not found: {os.environ['REPNET_DATA_DIR']} — edit REPNET_DATA_DIR above"
import src.train as train
print('data dir:', os.path.relpath(os.environ['REPNET_DATA_DIR']))
print('device:  ', train.device_name(train.resolve_device()))

## 1. Configure the run

`N_SPLITS_TO_RUN` controls how many patient-grouped splits to train. Each
split trains one model for up to 80 epochs, so the full release run
(`N_SPLITS_TO_RUN = 30`) can take **hours** on CPU/MPS.

- Start with a small number (e.g. `2`) to sanity-check the pipeline end to end.
- Set it to `train.N_SPLITS` (30) to reproduce the full released distribution.
- To cap per-split time, set `REPNET_TIME_BUDGET` (seconds/split) **before** the
  training cell, e.g. `os.environ['REPNET_TIME_BUDGET'] = '120'`. Note this is
  read by `src/train.py` at import time, so set it before importing if you
  need it to take effect (restart the kernel and set it in the setup cell).

In [ ]:
N_SPLITS_TO_RUN = 2   # set to train.N_SPLITS (30) for the full release run
OUT_DIR = os.path.join('results', 'multisplit_notebook')
print(f'will train {N_SPLITS_TO_RUN} split(s); artifacts -> {OUT_DIR}/')

## 2. Train

Calls `src.train.main` — the release training code — with the configured number
of splits. Per-split AUROC/AUPRC stream below as each split finishes.

In [ ]:
summary, rows = train.main(n_splits=N_SPLITS_TO_RUN, out_dir=OUT_DIR, write=True)
print()
print(f"AUROC {summary['auroc_mean']:.4f} +/- {summary['auroc_std']:.4f} | "
      f"AUPRC {summary['auprc_mean']:.4f} +/- {summary['auprc_std']:.4f}")
print(f"params={summary['num_params']:,} | "
      f"{summary['seconds_per_split_mean']:.0f}s/split on {summary['device']}")

## 3. Next steps

- **Evaluate the freshly trained checkpoints** by pointing `reproduce.ipynb`
  (its `RESULTS_DIR`) at the `OUT_DIR` above, or run `python -m src.evaluate`.
- **Generate the analysis figures** (split distribution, ROC/PR, calibration,
  lead importance, Lead-II saliency) with `python -m src.analyze`.

For the one-shot command-line equivalent of this notebook, run
`python -m src.train` (which trains all 30 splits with the release defaults).